In [0]:
import os

_classroom_setup_common_nb_path = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

_PROJECT_ROOT_DIR = "/Workspace" + os.path.dirname(
    os.path.dirname(_classroom_setup_common_nb_path)
)

In [0]:
%sql
-- Create safe function for catalog name
CREATE OR REPLACE TEMPORARY FUNCTION safe_uc_name(value STRING)
RETURNS STRING
RETURN
  COALESCE(
    NULLIF(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          LOWER(TRIM(value)),
          '[^a-z0-9_]',
          '_'
        ),
        '_+',
        '_'
      ),
      ''
    ),
    'user'
  );

-- Create temp view of all available catalogs
CREATE OR REPLACE TEMPORARY VIEW list_of_catalogs AS
SELECT LOWER(catalog_name) AS catalog_names
FROM system.information_schema.catalogs;

In [0]:
%sql
BEGIN

-- =========================================
-- 1. Declare variables
-- =========================================
  DECLARE user_email STRING;
  DECLARE user_name STRING;
  DECLARE safe_user_name STRING;
  DECLARE vocareum_catalog_name STRING DEFAULT 'NA';
  DECLARE catalog_name STRING DEFAULT 'NA';
  DECLARE catalog_already_exists BOOLEAN DEFAULT FALSE;
  DECLARE create_catalog_statement STRING DEFAULT '';
  DECLARE create_view_with_catalog_name STRING DEFAULT '';
  DECLARE test STRING DEFAULT '';

-- =========================================
-- 2. Capture current user
-- =========================================
  SET user_email = current_user();
  SET user_name = SPLIT(user_email, '@')[0];
  SET safe_user_name = (SELECT safe_uc_name(SPLIT(current_user(), '@')[0]));

-- ==================================================================================
-- 3. Determine Vocareum or Non-Vocareum workspace and set/create catalog
-- ==================================================================================
  IF (LOWER(user_email) LIKE '%@vocareum.com')
    THEN
      SET vocareum_catalog_name = safe_user_name;

      IF EXISTS (
        SELECT 1
        FROM list_of_catalogs
        WHERE catalog_names = LOWER(vocareum_catalog_name)
      ) THEN
        SET catalog_already_exists = TRUE;
        SET create_view_with_catalog_name = CONCAT(
              "CREATE OR REPLACE TEMP VIEW catalog_name_vw AS ",
              "SELECT '", vocareum_catalog_name, "' AS catalog_name"
            );
      END IF;

  ELSE
    SET catalog_name = CONCAT('labuser_', safe_user_name);
    SET create_view_with_catalog_name = CONCAT(
      "CREATE OR REPLACE TEMP VIEW catalog_name_vw AS ",
      "SELECT '", catalog_name, "' AS catalog_name"
    );

    IF EXISTS (
      SELECT 1
      FROM list_of_catalogs
      WHERE catalog_names = LOWER(catalog_name)
    )
      THEN
        SET catalog_already_exists = TRUE;
        SET test = 'catalog exists';
    ELSE
      SET create_catalog_statement = CONCAT(
        'CREATE CATALOG IF NOT EXISTS ',
        catalog_name
      );
      SET test = 'catalog does not exist';
      EXECUTE IMMEDIATE create_catalog_statement;
    END IF;

  END IF;

  EXECUTE IMMEDIATE create_view_with_catalog_name;

-- ===================================================================
-- 4. Return final results
-- ===================================================================
  SELECT
    CASE
      WHEN vocareum_catalog_name <> 'NA' THEN 'Vocareum Workspace'
      ELSE 'Non-Vocareum Workspace'
    END AS Workspace,
    CASE
      WHEN vocareum_catalog_name <> 'NA' THEN vocareum_catalog_name
      ELSE catalog_name
    END AS `User Catalog Name`,
    CASE
      WHEN catalog_already_exists = TRUE THEN 'Catalog Already Exists'
      ELSE 'Catalog Created'
    END AS `Catalog Status`;
END;

In [0]:
%sql
-- Declare and set the catalog variable from the temp view
DECLARE OR REPLACE my_catalog STRING;
SET VAR my_catalog = (SELECT catalog_name FROM catalog_name_vw);

-- Set the active catalog
USE CATALOG IDENTIFIER(my_catalog);

In [0]:
my_catalog = spark.sql("SELECT catalog_name FROM catalog_name_vw").first()[0]

# The temporary views and function used to resolve the catalog are no longer
# needed once my_catalog is captured. Drop them so they don't linger in the
# session.
spark.sql("DROP VIEW IF EXISTS catalog_name_vw")
spark.sql("DROP VIEW IF EXISTS list_of_catalogs")
spark.sql("DROP TEMPORARY FUNCTION IF EXISTS safe_uc_name")

In [0]:
SCHEMA_NAME = "vintage_audio"
VOLUME_NAME = "data"

# Create the schema and set the session context explicitly against the resolved
# catalog. We qualify with `my_catalog` rather than relying on the current
# session catalog, which in some environments (dbacademy/Vocareum) can revert
# to a default like 'dbacademy'.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{my_catalog}`.`{SCHEMA_NAME}`")
spark.sql(f"USE CATALOG `{my_catalog}`")
spark.sql(f"USE SCHEMA `{SCHEMA_NAME}`")

## Create volume inside the schema
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{my_catalog}`.`{SCHEMA_NAME}`.`{VOLUME_NAME}`")

In [0]:
########################################
# Foundation: stage the raw CSV files into the volume (idempotent).
#
# Every notebook in the course runs this Common setup, so staging the CSVs
# here guarantees the raw files are present no matter which notebook a learner
# opens first. Copies are skipped when the file already exists, so on a normal
# run this is just a handful of existence checks (no copying).
########################################
import os
import shutil

_VOLUME_BASE_PATH = f"/Volumes/{my_catalog}/{SCHEMA_NAME}/{VOLUME_NAME}"

# All CSVs shipped with the course. Source files become base tables; order
# files are loaded in M06/M07.
_ALL_CSV_FILES = [
    "customers_raw", "products_raw", "stores_raw",
    "orders_2024_q1", "orders_2024_q2", "orders_2024_q3", "orders_2024_q4",
    "orders_2025_q1", "orders_2025_q2",
]


def _stage_csv_files() -> None:
    """Copy every course CSV from the workspace bundle into the data volume,
    skipping any file that is already present. Idempotent and silent on a
    normal run (no copies needed)."""
    src_dir = os.path.join(
        _PROJECT_ROOT_DIR,
        "Get Started with Databricks for Data Warehousing",
        "Includes", "data",
    )
    for stem in _ALL_CSV_FILES:
        src_path    = os.path.join(src_dir, f"{stem}.csv")
        target_path = f"{_VOLUME_BASE_PATH}/{stem}.csv"
        if os.path.exists(target_path):
            continue
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"Course data file not found in bundle: {src_path}")
        shutil.copy(src_path, target_path)


_stage_csv_files()

In [0]:
# Final re-assertion of the session context, so unqualified table references in
# the notebook (e.g. SELECT ... FROM orders_bronze) resolve to the learner's
# catalog and the vintage_audio schema regardless of any session drift above.
spark.sql(f"USE CATALOG `{my_catalog}`")
spark.sql(f"USE SCHEMA `{SCHEMA_NAME}`")